In [1]:
import os
import findspark
findspark.init()

from pyspark.sql import SparkSession

# 1. إعدادات البيئة (لازم تكون قبل بناء السيشين)
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-19"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 2. بناء السيشين بإعدادات الـ Victus
spark = SparkSession.builder \
    .master("local[4]") \
    .appName("NYC-Taxi-Victus-Final") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.memory", "3g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.sql.broadcastTimeout", "600") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"Spark {spark.version} is up and running correctly!")
spark.range(3).show()

Spark 4.1.1 is up and running correctly!
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



In [2]:
# تأكد من تعديل المسار لمكان الملف على جهازك
data_path = r"C:\Users\HP\Downloads\final_cleaned_data.parquet"
# قراءة الداتا بـ Schema جاهزة
df_ready = spark.read.parquet(data_path)

print(f"Data Loaded! Total Rows: {df_ready.count()}")

Data Loaded! Total Rows: 10803240


In [3]:
# Cell (3) - Optimized Conversion
print("Memory-Safe Conversion to Pandas...")

# اختيار الأعمدة الضرورية فقط قبل التحويل
selected_df = df_ready.select('trip_distance', 'passenger_count', 'hour', 'day_of_week', 'fare_amount')

# التحويل لبانداز (هنا مش هيحصل Crash بإذن الله)
pdf = selected_df.toPandas()

X = pdf[['trip_distance', 'passenger_count', 'hour', 'day_of_week']]
y = pdf['fare_amount']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(" Conversion Success!")

Memory-Safe Conversion to Pandas...
 Conversion Success!


In [4]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

print("Training on NVIDIA GPU...")
model_xgb = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    tree_method='hist', # خوارزمية سريعة جداً
    device='cuda',      # هنا السحر.. استخدام الـ GPU
    random_state=42
)

model_xgb.fit(X_train, y_train)
print("GPU Training Completed!")

Training on NVIDIA GPU...
GPU Training Completed!


In [5]:
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. التوقع
y_pred = model_xgb.predict(X_test)

# 2. حساب الـ RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f" Final Model RMSE: {rmse:.4f}")

# 3. حفظ الموديل بالطريقة الأضمن (Booster Save)
try:
    model_xgb.get_booster().save_model("nyc_taxi_model_victus.json")
    print("Model saved as 'nyc_taxi_model_victus.json'")
except Exception as e:
    print(f"❌ Error saving model: {e}")

# 4. عرض عينة سريعة للتأكد
import pandas as pd
comparison = pd.DataFrame({'Real Fare': y_test, 'Predicted Fare': y_pred}).head(5)
print("\nQuick Look at Predictions:")
print(comparison)

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:774: UserWarning: [19:39:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


 Final Model RMSE: 3.1251
Model saved as 'nyc_taxi_model_victus.json'

Quick Look at Predictions:
         Real Fare  Predicted Fare
3453832       34.0       21.008232
9649774       13.5       12.576935
1412502       34.0       35.687477
2263922       12.5       14.672922
4448040        8.5        6.186289


In [7]:
import os
print("📍 المسار الكامل للموديل هو:")
print(os.path.abspath("nyc_taxi_model_victus.json"))

📍 المسار الكامل للموديل هو:
c:\Users\HP\AppData\Local\Programs\Microsoft VS Code\nyc_taxi_model_victus.json
